# RadioML 2016.10A — Dataset Exploration

This notebook loads and explores the RadioML 2016.10A dataset, which is the primary
benchmark dataset for automatic modulation classification (AMC) research.

**Source:** DeepSig Inc. — https://www.deepsig.ai/datasets  
**Citation:** T. J. O'Shea, J. Corgan, and T. C. Clancy, "Convolutional Radio Modulation
Recognition Networks," in Proc. International Conference on Engineering Applications of
Neural Networks, 2016.

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = "../../Data/RML2016.10a_dict.pkl"  # See ../../Data/README.md

with open(DATA_PATH, "rb") as f:
    data = pickle.load(f, encoding="latin1")

print(f"Type: {type(data)}")
print(f"Number of keys: {len(data)}")
print(f"Key format: (modulation_type, snr_dB)")
print(f"Example key: {list(data.keys())[0]}")

In [ ]:
# Extract unique modulation types and SNR levels
mod_types = sorted(set(k[0] for k in data.keys()))
snr_levels = sorted(set(k[1] for k in data.keys()))

print(f"Modulation types ({len(mod_types)}):")
for i, m in enumerate(mod_types):
    category = "Digital" if m in ["BPSK","QPSK","8PSK","QAM16","QAM64","CPFSK","GFSK","PAM4"] else "Analog"
    print(f"  {i+1:2d}. {m:10s}  ({category})")

print(f"\nSNR levels ({len(snr_levels)}): {snr_levels} dB")
print(f"Range: {min(snr_levels)} dB to {max(snr_levels)} dB")

In [ ]:
# Dataset statistics
sample_shape = data[list(data.keys())[0]].shape
samples_per_key = sample_shape[0]
total_samples = sum(v.shape[0] for v in data.values())

print(f"Sample shape per key: {sample_shape}")
print(f"  → {samples_per_key} samples per (modulation, SNR) pair")
print(f"  → Each sample: {sample_shape[1]} channels (I and Q) × {sample_shape[2]} time steps")
print(f"\nTotal samples in dataset: {total_samples:,}")
print(f"Total keys: {len(mod_types)} modulations × {len(snr_levels)} SNRs = {len(mod_types)*len(snr_levels)}")

In [ ]:
# Visualize example IQ waveforms for selected modulations
example_mods = ["BPSK", "QPSK", "QAM16", "AM-DSB", "WBFM"]
example_snr = 10  # dB — clean enough to see structure

fig, axes = plt.subplots(len(example_mods), 2, figsize=(14, 3 * len(example_mods)))

for i, mod in enumerate(example_mods):
    sample = data[(mod, example_snr)][0]  # First sample
    I, Q = sample[0], sample[1]

    # Time-domain I/Q
    axes[i, 0].plot(I, label="I (In-phase)", alpha=0.8)
    axes[i, 0].plot(Q, label="Q (Quadrature)", alpha=0.8)
    axes[i, 0].set_title(f"{mod} @ {example_snr} dB — I/Q Waveform")
    axes[i, 0].legend(loc="upper right", fontsize=8)
    axes[i, 0].set_ylabel("Amplitude")

    # IQ scatter (constellation-like)
    axes[i, 1].scatter(I, Q, s=5, alpha=0.6)
    axes[i, 1].set_title(f"{mod} @ {example_snr} dB — IQ Scatter")
    axes[i, 1].set_xlabel("I")
    axes[i, 1].set_ylabel("Q")
    axes[i, 1].set_aspect("equal")

axes[-1, 0].set_xlabel("Time Step")
plt.tight_layout()
plt.savefig("../../Results/radioml_example_waveforms.png", dpi=150)
plt.show()
print("Saved → ../../Results/radioml_example_waveforms.png")

In [ ]:
# Show how signal quality degrades with SNR
mod = "QPSK"
snrs_to_show = [-10, 0, 10, 18]

fig, axes = plt.subplots(1, len(snrs_to_show), figsize=(16, 3.5))
for i, snr in enumerate(snrs_to_show):
    sample = data[(mod, snr)][0]
    axes[i].scatter(sample[0], sample[1], s=8, alpha=0.6)
    axes[i].set_title(f"{mod} @ {snr} dB")
    axes[i].set_xlabel("I")
    axes[i].set_ylabel("Q")
    axes[i].set_aspect("equal")

plt.suptitle("Effect of SNR on QPSK Constellation", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("../../Results/snr_effect_on_constellation.png", dpi=150)
plt.show()
print("Saved → ../../Results/snr_effect_on_constellation.png")

## Summary

| Property | Value |
|----------|-------|
| Modulation types | 11 (8 digital + 3 analog) |
| SNR range | −20 dB to +18 dB (20 levels, 2 dB steps) |
| Samples per (mod, SNR) | 1,000 |
| Sample format | 2 × 128 (I/Q × time steps) |
| Total samples | 220,000 |

This dataset is the de facto standard for AMC benchmarking. For our project, we will
train on a subset of classes and evaluate open-set rejection on held-out classes.